In [3]:
###Load packages###
import pandas as pd
import os
import ast
from scipy import stats
from matplotlib import pyplot as plt
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
import numpy as np
import statsmodels.formula.api as smf
import seaborn as sns


###Load cleaned dataset###

#Set file paths
topdir = '/Users/sm6511/Desktop/Prediction-Accomodation-Exp'
study = 'Study5.0'
cleandir = os.path.join(topdir, f'data/{study}/Cleaned')
outputdir = os.path.join(topdir, f'Analysis/{study}')
outputdirCombined = os.path.join(topdir, f'data/Combined')
outputdirCleaned = os.path.join(topdir, f'data/{study}/Cleaned')
os.makedirs(outputdir, exist_ok=True)

#Read in cleaned data 
accomodate_path = os.path.join(cleandir, f'{study}Accommodate.csv')
predict_path   = os.path.join(cleandir, f'{study}Predict.csv')

df_accommodate = pd.read_csv(accomodate_path)
df_predict   = pd.read_csv(predict_path)

df_accommodate['task'] = 'accommodate'
df_predict['task']   = 'predict'


print("Accommodate columns:", df_accommodate.columns.tolist())
print("Predict columns:", df_predict.columns.tolist())


Accommodate columns: ['participant', 'free_texts', 'feedback', 'fertility_score', 'trial_stop_time', 'testing_image_order', 'testing_responses', 'training_categories', 'training_feet', 'training_stripes', 'testing_categories', 'conditionOrder', 'training_image_order', 'attention_check', 'cogpro_predict', 'cogpro_explain', 'relevant_dim', 'irrelevant_dim', 'feet_high', 'stripes_low', 'stripes_high', 'feet_low', 'feet_discrete_slider.response', 'feet_direction_slider.response', 'feet_continuous_slider.response', 'feet_discrete_slider_mid.response', 'feet_direction_slider_mid.response', 'feet_continuous_slider_mid.response', 'stripes_discrete_slider.response', 'stripes_direction_slider.response', 'stripes_continuous_slider.response', 'stripes_discrete_slider_mid.response', 'stripes_direction_slider_mid.response', 'stripes_continuous_slider_mid.response', 'task']
Predict columns: ['participant', 'training_responses', 'fertility_score', 'error', 'feedback', 'trial_stop_time', 'testing_image

In [4]:
#Converting string representations of lists back to lists

def parse_list_column(x):
    """take column entries that are strings representing lists and convert them to actual lists"""
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if x.startswith('[') and x.endswith(']'):
            return ast.literal_eval(x)
        else:
            return [x]
    return []
for col in ['training_feet', 'training_stripes', 'training_image_order', 'training_categories', 'testing_categories']:
    df_accommodate[col] = df_accommodate[col].apply(parse_list_column)
    df_predict[col]   = df_predict[col].apply(parse_list_column)

df_accommodate['testing_responses'] = df_accommodate['testing_responses'].apply(ast.literal_eval)
df_accommodate['fertility_score'] = df_accommodate['fertility_score'].apply(ast.literal_eval)
df_accommodate['testing_image_order'] = df_accommodate['testing_image_order'].apply(ast.literal_eval)
df_predict['testing_responses'] = df_predict['testing_responses'].apply(ast.literal_eval)
df_predict['fertility_score'] = df_predict['fertility_score'].apply(ast.literal_eval)
df_predict['testing_image_order'] = df_predict['testing_image_order'].apply(ast.literal_eval)
#Combine the dataframes and create an arbitrary column for participant numbering (the yoked orders are already stored in 'conditionOrder')
df_combined = pd.concat([df_accommodate, df_predict], ignore_index=True)
df_combined['participant'] = range(1, len(df_combined) + 1)



In [5]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

all_participant_rows = []

for suffix, time in [("_mid", "mid"), ("", "end")]:

    for _, row in df_combined.iterrows():

        feet_yes = int(
            str(row[f"feet_discrete_slider{suffix}.response"]).strip().lower()
            == "yes"
        )

        stripes_yes = int(
            str(row[f"stripes_discrete_slider{suffix}.response"]).strip().lower()
            == "yes"
        )

        model_param_score = feet_yes + stripes_yes

        all_participant_rows.append({
            "participant": row["participant"],
            "task": row["task"],
            "time": time,
            "model_param_score": model_param_score,
            "conditionOrder": row["conditionOrder"],

            "relevant_dim": row["relevant_dim"],
            "irrelevant_dim": row["irrelevant_dim"],

            "feet_high": row["feet_high"],
            "stripes_high": row["stripes_high"],
            "feet_low": row["feet_low"],
            "stripes_low": row["stripes_low"],

            "feet_response":
                row[f"feet_discrete_slider{suffix}.response"],
            "stripes_response":
                row[f"stripes_discrete_slider{suffix}.response"],

            "feet_reported_relevant": feet_yes,
            "stripes_reported_relevant": stripes_yes,

            "cogpro_explain": row["cogpro_explain"],
            "cogpro_predict": row["cogpro_predict"],

            # Binary outcome: 1 when the participant selected both features
            "overfit": int(model_param_score == 2)
        })

df_params_all = pd.DataFrame(all_participant_rows)

df_params_all.to_csv(os.path.join(outputdirCombined, f'df_params_study5_for_r.csv'), index=False)



In [8]:
print(len(df_accommodate))

292


In [7]:
contingency = pd.crosstab(
    df_params_all['task'],
    df_params_all['overfit']
)

print(contingency)

chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-square = {chi2:.3f}")
print(f"df = {dof}")
print(f"p-value = {p:.4f}")

overfit        0    1
task                 
accommodate  238  346
predict      230  354
Chi-square = 0.175
df = 1
p-value = 0.6760


In [10]:
contingency_all = pd.crosstab(
    df_params_all['task'],
    df_params_all['model_param_score']
)

print(contingency_all)

chi2, p, dof, expected = chi2_contingency(contingency_all)

print(f"Chi-square = {chi2:.3f}")
print(f"df = {dof}")
print(f"p-value = {p:.4f}")

model_param_score   0    1    2
task                           
accommodate        50  188  346
predict            43  187  354
Chi-square = 0.621
df = 2
p-value = 0.7331


In [5]:
#Create map from short codes to feature descriptions

feet_map = {
    'webbed': 'f',
    'curved/pointy': 'c'
}

stripes_map = {
    'up/right': 'e',
    'up/left': 'w'
}



feature_maps = {
    'feet': feet_map,
    'stripes': stripes_map
}


In [6]:
#Compute feature importance scores

from doctest import debug


def compute_feature_importance_from_df(df):
    """
    Compute numeric feature importance scores(-7 to 7) for each participant,
    based on the saved slider_responses and the feature _high/_low mapping.
    This is computed based on whether a feature was really relevant (positive sign) or irrelevant (negative sign).
    0 = no response or feature was not thought to be relevant
    columns:
      - wings_discrete_slider.response, wings_direction_slider.response, wings_continuous_slider.response
      - color_discrete_slider.response, ...
      - tail_discrete_slider.response, ...
      - wings_high, wings_low, color_high, color_low, tail_high, tail_low
    """
    features = ['feet', 'stripes']
    def compute_row_importance(row, feat, mid):
        if mid:
            suffix = '_mid'
        else:
            suffix = ''
        disc = row[f'{feat}_discrete_slider{suffix}.response']
        dirc = row[f'{feat}_direction_slider{suffix}.response']
        cont = row[f'{feat}_continuous_slider{suffix}.response']

        #If they said a feature wasn't relevant, then importance is 0
        
        if disc == 'No' or pd.isna(disc):
            return 0.0
        
        # Make sure continuous slider value exists, if not, set it to 0
        cont_val = float(cont) if not pd.isna(cont) else 0.0

        # Get mapping from long to short feature name
        mapping = feature_maps.get(feat, {})

        # Normalize strings: strip whitespace, collapse multiple spaces, lower-case
        def normalize_str(s):
            """Strip leading/trailing whitespace and collapse internal multiple spaces."""

            if isinstance(s, str):
                return " ".join(s.split()).lower()
            return ""
        

        #Name of features need to be normalized for comparison using the mapping
        dirc_norm = normalize_str(dirc)

        #Get internal short code for selected feature direction
        internal_dirc = mapping.get(dirc_norm, None)
        

        high_val = normalize_str(row[f'{feat}_high'])
        low_val  = normalize_str(row[f'{feat}_low'])
        

        
        # Debug print statement (make sure mappings look right)
        debug = True
        if debug:
            print('response:', repr(dirc_norm), 'internal:', repr(internal_dirc), 
                'high:', repr(high_val), 'low:', repr(low_val))
            

        #If they correctly selected the high feature, assign positive sign
        if internal_dirc == high_val:
            sign = 1
        #If they incorrectly selected the low feature, assign negative sign
        elif internal_dirc == low_val:
            if debug:
                print('in negative')
            sign = -1
        else:
            if debug:
                print('in empty')
            sign = 0
            cont_val = 0.0

        # Add sign to continuous value
        importance = cont_val * sign

        return importance

    
    # Compute for each feature
    for feat in features:
        df[f'{feat}_importance'] = df.apply(lambda row: compute_row_importance(row, feat, mid=False), axis=1)
        df[f'{feat}_importance_mid'] = df.apply(lambda row: compute_row_importance(row, feat, mid=True), axis=1)
    
    return df

df_combined = compute_feature_importance_from_df(df_combined)
#df_filtered = compute_feature_importance_from_df(df_filtered)
if debug:
    print(df_combined['feet_importance'])

response: 'curved/pointy' internal: 'c' high: 'c' low: 'f'
response: 'webbed' internal: 'f' high: 'f' low: 'c'
response: 'webbed' internal: 'f' high: 'f' low: 'c'
response: 'curved/pointy' internal: 'c' high: 'c' low: 'f'
response: 'curved/pointy' internal: 'c' high: 'c' low: 'f'
response: 'webbed' internal: 'f' high: 'f' low: 'c'
response: 'curved/pointy' internal: 'c' high: 'c' low: 'f'
response: 'webbed' internal: 'f' high: 'c' low: 'f'
in negative
response: 'webbed' internal: 'f' high: 'f' low: 'c'
response: 'curved/pointy' internal: 'c' high: 'c' low: 'f'
response: 'webbed' internal: 'f' high: 'f' low: 'c'
response: 'webbed' internal: 'f' high: 'f' low: 'c'
response: 'webbed' internal: 'f' high: 'f' low: 'c'
response: 'webbed' internal: 'f' high: 'f' low: 'c'
response: 'curved/pointy' internal: 'c' high: 'c' low: 'f'
response: 'curved/pointy' internal: 'c' high: 'c' low: 'f'
response: 'curved/pointy' internal: 'c' high: 'c' low: 'f'
response: 'webbed' internal: 'f' high: 'f' low: 

In [7]:
import pandas as pd
"""Reshape to long format with 1 row per participant x feature dimension"""
# Keep only necessary columns
cols_to_keep = [
    'participant', 'task', 
    'feet_importance', 'feet_importance_mid', 'stripes_importance', 'stripes_importance_mid',
    'relevant_dim', 'irrelevant_dim', 'feet_high','stripes_high', 'feet_discrete_slider.response',
    'cogpro_explain', 'cogpro_predict'
]

df_long = df_combined[cols_to_keep].copy()

df_long = df_long.melt(
    id_vars=[
        'participant', 'task', 'relevant_dim', 'irrelevant_dim',
        'stripes_high', 'feet_high', 'feet_discrete_slider.response',
        'cogpro_explain', 'cogpro_predict'
    ],
    value_vars=[
        'feet_importance', 'feet_importance_mid',
        'stripes_importance', 'stripes_importance_mid'
    ],
    var_name='feature_dimension',
    value_name='feature_importance'
)

# create time column
df_long['time'] = np.where(
    df_long['feature_dimension'].str.endswith('_mid'),
    'mid',
    'end'
)

# clean feature dimension back to feet/stripes
df_long['feature_dimension'] = (
    df_long['feature_dimension']
    .str.replace('_importance_mid', '', regex=False)
    .str.replace('_importance', '', regex=False)
)

df_long['feature_relevance'] = np.where(
    df_long['feature_dimension'] == df_long['relevant_dim'],
    'relevant',
    'irrelevant'
)

print(df_long.tail(20))

      participant     task relevant_dim irrelevant_dim stripes_high feet_high  \
2316          565  predict      stripes           feet            E         F   
2317          566  predict      stripes           feet            W         C   
2318          567  predict         feet        stripes            E         C   
2319          568  predict         feet        stripes            E         F   
2320          569  predict      stripes           feet            E         F   
2321          570  predict      stripes           feet            E         C   
2322          571  predict      stripes           feet            E         C   
2323          572  predict         feet        stripes            E         F   
2324          573  predict         feet        stripes            E         C   
2325          574  predict      stripes           feet            W         C   
2326          575  predict      stripes           feet            W         F   
2327          576  predict  

In [14]:
print(df_long)
df_long.to_csv(os.path.join(outputdirCombined, 'df_long_for_R-Study5.csv'), index=False)

      participant         task relevant_dim irrelevant_dim stripes_high  \
0               1  accommodate      stripes           feet            W   
1               2  accommodate      stripes           feet            W   
2               3  accommodate         feet        stripes            W   
3               4  accommodate         feet        stripes            W   
4               5  accommodate         feet        stripes            E   
...           ...          ...          ...            ...          ...   
2331          580      predict         feet        stripes            W   
2332          581      predict      stripes           feet            W   
2333          582      predict         feet        stripes            W   
2334          583      predict         feet        stripes            E   
2335          584      predict      stripes           feet            W   

     feet_high feet_discrete_slider.response  cogpro_explain  cogpro_predict  \
0            C     

In [8]:
#Group by average food amount per item in training
df = df_combined[['task', 'feet_high', 'stripes_high', 'training_image_order', 'fertility_score', 'conditionOrder']]
df_long2 = (
    df
    .explode(['training_image_order', 'fertility_score'])
    .rename(columns={'training_image_order': 'item'})
)
avg_food = (
    df_long2
    .groupby(['task', 'conditionOrder', 'item'], as_index=False)
    ['fertility_score']
    .mean()
)
avg_food_train = avg_food.copy()
print(avg_food_train.head(20))

avg_food_2 = (
    df_long2
    .groupby(['task', 'conditionOrder', 'item', 'feet_high', 'stripes_high'], as_index=False)
    ['fertility_score']
    .mean()
)
print(avg_food_2.head(20))


           task  conditionOrder item fertility_score
0   accommodate               1  E_C             7.5
1   accommodate               1  E_F             4.5
2   accommodate               1  W_C             6.0
3   accommodate               1  W_F             2.5
4   accommodate               2  E_C            2.25
5   accommodate               2  E_F             2.5
6   accommodate               2  W_C            8.25
7   accommodate               2  W_F            6.25
8   accommodate               3  E_C            3.75
9   accommodate               3  E_F             7.5
10  accommodate               3  W_C            4.75
11  accommodate               3  W_F            8.25
12  accommodate               4  E_C            6.75
13  accommodate               4  E_F             8.0
14  accommodate               4  W_C             4.5
15  accommodate               4  W_F             4.5
16  accommodate               5  E_C             6.5
17  accommodate               5  E_F          

In [9]:
#Get food consumption ratings by item

df = df_combined[['task', 'conditionOrder', 'testing_image_order', 'testing_responses',
                  'relevant_dim', 'irrelevant_dim', 'stripes_high', 'feet_high']]
df_long2 = (
    df
    .explode(['testing_image_order', 'testing_responses'])
    .rename(columns={'testing_image_order': 'item'})
)
avg_food_test = df_long2.copy()
print(avg_food_test)


            task  conditionOrder item testing_responses relevant_dim  \
0    accommodate             226  W_F               6.0      stripes   
0    accommodate             226  W_C               6.0      stripes   
0    accommodate             226  E_F               6.0      stripes   
0    accommodate             226  E_C               6.0      stripes   
1    accommodate             210  W_C               9.0      stripes   
..           ...             ...  ...               ...          ...   
582      predict             265  W_F               1.0         feet   
583      predict              86  W_F               9.0      stripes   
583      predict              86  E_F               2.0      stripes   
583      predict              86  W_C               8.0      stripes   
583      predict              86  E_C               3.0      stripes   

    irrelevant_dim stripes_high feet_high  
0             feet            W         C  
0             feet            W         C  
0  

In [10]:
#Now merge the two (actual food amounts in training vs ratings in testing) and compute error
df_merged = avg_food_test.merge(
    avg_food_train,
    on=['task', 'conditionOrder', 'item'],
    how='left'
)

#Add Error and absolute error
df_merged['error'] = (
    df_merged['testing_responses'] - df_merged['fertility_score']
)
df_merged['abs_error'] = df_merged['error'].abs()
df_merged[['stripes', 'feet']] = df_merged['item'].str.split('_', expand=True)

print(df_merged)

             task  conditionOrder item testing_responses relevant_dim  \
0     accommodate             226  W_F               6.0      stripes   
1     accommodate             226  W_C               6.0      stripes   
2     accommodate             226  E_F               6.0      stripes   
3     accommodate             226  E_C               6.0      stripes   
4     accommodate             210  W_C               9.0      stripes   
...           ...             ...  ...               ...          ...   
2331      predict             265  W_F               1.0         feet   
2332      predict              86  W_F               9.0      stripes   
2333      predict              86  E_F               2.0      stripes   
2334      predict              86  W_C               8.0      stripes   
2335      predict              86  E_C               3.0      stripes   

     irrelevant_dim stripes_high feet_high fertility_score error abs_error  \
0              feet            W         C   

In [11]:
df_merged.to_csv(os.path.join(outputdirCombined, 'df_merged_for_R_Study5.csv'), index=False)

In [12]:
# Columns indicating whether the item's feature is the "high" dimension (1 or 0 coding)
df_merged['stripes_match_high']  = (df_merged['stripes']  == df_merged['stripes_high']).astype(int)
df_merged['feet_match_high'] = (df_merged['feet'] == df_merged['feet_high']).astype(int)
#print(df_merged.head(20))
# Group by participant
participant_corrs = []

for pid, g in df_merged.groupby(['task', 'conditionOrder']):
    for feat in ['stripes','feet']:
        # Column indicating match to high value
        match_col = f"{feat}_match_high"
        
        # Compute correlation
        corr = g['testing_responses'].corr(g[match_col])
        
        # Determine if this feature is relevant for this participant
        relevant = g['relevant_dim'].iloc[0] == feat
        high_col  = f"{feat}_high"
        
        # Store
        participant_corrs.append({
            'participant': pid,
            'task': g['task'].iloc[0],
            'feature_dimension': feat,
            'high_value': g[high_col].iloc[0],
            'feature_relevance': 'relevant' if relevant else 'irrelevant',
            'correlation': corr,
            'irrelevant_dim': g['irrelevant_dim'].iloc[0],
            'abs_correlation': abs(corr) if pd.notna(corr) else None
        })

df_corr = pd.DataFrame(participant_corrs)
print(df_corr.tail(40))



         participant     task feature_dimension high_value feature_relevance  \
1128  (predict, 280)  predict           stripes          E          relevant   
1129  (predict, 280)  predict              feet          F        irrelevant   
1130  (predict, 281)  predict           stripes          E        irrelevant   
1131  (predict, 281)  predict              feet          C          relevant   
1132  (predict, 282)  predict           stripes          W          relevant   
1133  (predict, 282)  predict              feet          F        irrelevant   
1134  (predict, 283)  predict           stripes          W        irrelevant   
1135  (predict, 283)  predict              feet          F          relevant   
1136  (predict, 285)  predict           stripes          E        irrelevant   
1137  (predict, 285)  predict              feet          C          relevant   
1138  (predict, 286)  predict           stripes          W          relevant   
1139  (predict, 286)  predict           

/opt/miniconda3/envs/PredictProj/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/miniconda3/envs/PredictProj/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [13]:
# Get the actual correlations between fertility score and feature match
# for each participant in training

actual_corrs = []

for (task, condition_order), g in df_merged.groupby(
    ["task", "conditionOrder"]
):

    participant = (task, condition_order)
    relevant_dim = g["relevant_dim"].iloc[0]

    for feat in ["stripes", "feet"]:

        match_col = f"{feat}_match_high"

        corr = g["fertility_score"].corr(g[match_col])

        actual_corrs.append({
            "participant": participant,
            "task": task,
            "conditionOrder": condition_order,
            "feature_dimension": feat,
            "feature_relevance": (
                "relevant"
                if feat == relevant_dim
                else "irrelevant"
            ),
            "actual_correlation": corr
        })

actual_corrs = pd.DataFrame(actual_corrs)

df_corr = df_corr.merge(
    actual_corrs[
        [
            "participant",
            "feature_dimension",
            "actual_correlation"
        ]
    ],
    on=["participant", "feature_dimension"],
    how="left"
)

print(df_corr.head(20))

          participant         task feature_dimension high_value  \
0    (accommodate, 1)  accommodate           stripes          E   
1    (accommodate, 1)  accommodate              feet          C   
2    (accommodate, 2)  accommodate           stripes          W   
3    (accommodate, 2)  accommodate              feet          C   
4    (accommodate, 3)  accommodate           stripes          W   
5    (accommodate, 3)  accommodate              feet          F   
6    (accommodate, 4)  accommodate           stripes          E   
7    (accommodate, 4)  accommodate              feet          F   
8    (accommodate, 5)  accommodate           stripes          W   
9    (accommodate, 5)  accommodate              feet          C   
10   (accommodate, 6)  accommodate           stripes          W   
11   (accommodate, 6)  accommodate              feet          F   
12   (accommodate, 7)  accommodate           stripes          W   
13   (accommodate, 7)  accommodate              feet          

In [14]:
df_corr.to_csv(os.path.join(outputdirCombined, 'df_corr_for_R_Study5.csv'), index=False)